### Import

In [3]:
import sys
import time
from time import perf_counter
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import mlflow
import optuna

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import root_mean_squared_log_error

from src.evaluation.metrics_report import evaluate_model
from src.models.train import cross_validate

d:\reps\rossmann\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv("../data/processed/rossmannV2.csv")

In [5]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("rossmann-forecasting")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1787652399876, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787652399876, lifecycle_stage='active', name='rossmann-forecasting', tags={}, trace_location=None, workspace='default'>

---

### Splitting the data and Metric choosing

In [6]:
df = df.sort_values("Date").reset_index(drop=True)

cutoff = "2015-01-01"

train = df[df["Date"] < cutoff].copy()
test = df[df["Date"] >= cutoff].copy()

In [7]:
X_train = train.drop(columns=["Sales", "Date"])
y_train = train["Sales"]

X_test = test.drop(columns=["Sales", "Date"])
y_test = test["Sales"]

In [8]:
tscv = TimeSeriesSplit(
    n_splits=5,
    gap=7
)

Here i added gap for so i have 7 rows gap between splits but it'll split wrong several times since dataset has several stores.

In [9]:
for train_idx, valid_idx in tscv.split(X_train):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_valid = X_train.iloc[valid_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_valid = y_train.iloc[valid_idx]

Prevents future data leaking.

---

I decided to use `RMSLE` as a metric for final model because it's care about relative difference rather than absolute difference. 

E.g. difference between 100 - 200 and 1000 - 1100 are the same on paper, but not in reality (100% diff VS 10%). 

In our case this metric actually recognizes that the first error in example is more significant in relative terms, so I'll use it.

---

For the model version comparison I'll use `MAE, RMSE and RMSLE` because they answer different types of questions which is:
- `MAE:` How many unit sales am i wrong on average?

- `RMSE:` How bad are my largest errors?

- `RMSLE:` How good am i at predicting relative sales levels?

---

### Baseline and Model comparison

In [10]:
ridge_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("model", Ridge(alpha=1.0))
])

In [11]:
start = time.perf_counter()

ridge_pipeline.fit(X_fold_train, y_fold_train)

ridge_predict = ridge_pipeline.predict(X_fold_valid)

# Sales can't be negative, so i clip predictions at 0 before computing RMSLE
ridge_predict = np.clip(ridge_predict, 0, None)

print(evaluate_model(y_fold_valid, ridge_predict))
print(f"Time: {perf_counter() - start:.4f}s.")

{'MAE': 826.2660765644697, 'RMSE': 1171.0625652210192, 'RMSLE': 1.9011912212526134}
Time: 1.9464s.


---

In [12]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=44
)

start = time.perf_counter()

xgb_model.fit(X_fold_train, y_fold_train)

xgb_predict = xgb_model.predict(X_fold_valid)

print(evaluate_model(y_fold_valid, xgb_predict))
print(f"Time: {perf_counter() - start:.4f}s.")

{'MAE': 446.8831481933594, 'RMSE': 674.0966186523438, 'RMSLE': 1.325823426246643}
Time: 14.6753s.


---

In [13]:
lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=46,
    verbose=-1
)

start = time.perf_counter()

lgbm_model.fit(X_fold_train, y_fold_train)

lgbm_predict = lgbm_model.predict(X_fold_valid)

print(evaluate_model(y_fold_valid, lgbm_predict))
print(f"Time: {perf_counter() - start:.4f}s.")

{'MAE': 458.4409893311257, 'RMSE': 695.1642749861792, 'RMSLE': 1.1521239970014348}
Time: 13.3915s.


Here's the conclusions based on the results:
- XGBoost and LGBM did almost `50% better` than baseline, but Ridge was 5-8 times faster than complex models

- LGBM finished work `45% faster` than XGB

- On the other side XGB did `6% better than LGBM on average`, which is not significant difference because those results are absolute

What's important is RMSLE results. LGBM actually did better on this one. It has lower logarithmic error than the second boosting model(`1.15 instead of 1.32`) which on percentage will be `216% instead of 274%`.

Because of LGBM speed, high scores and balanced tradeoff I'll use it as a final model.

---

### Hyperparams + Best model

In [14]:
def objective(trial):
    max_depth = trial.suggest_int("max_depth", 4, 10)
    num_leaves = trial.suggest_int("num_leaves", 2 ** (max_depth - 1), 2 ** max_depth)

    params = {
        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 300, 800, step=50),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "random_state": 12,
        "verbose": -1,
        "n_jobs": -1,
    }

    model = LGBMRegressor(**params)
    scores = cross_validate(model, X_train, y_train, n_splits=5, gap=7)
    return float(scores["RMSLE"].mean())

In [15]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=12))
study.optimize(objective, n_trials=30)

[I 2026-08-27 16:35:58,097] A new study created in memory with name: no-name-35710e0c-e495-4dd8-a726-271369c6d73b


[I 2026-08-27 16:37:00,885] Trial 0 finished with value: 1.3101856740643845 and parameters: {'max_depth': 5, 'num_leaves': 28, 'learning_rate': 0.02200800788928047, 'n_estimators': 550, 'colsample_bytree': 0.5072874812427098, 'subsample': 0.9593735040499425, 'reg_alpha': 4.007369758482047, 'reg_lambda': 0.0013604597909008316, 'min_child_samples': 193, 'min_split_gain': 0.13720932135607644}. Best is trial 0 with value: 1.3101856740643845.
[I 2026-08-27 16:38:19,996] Trial 1 finished with value: 1.5256901874812818 and parameters: {'max_depth': 5, 'num_leaves': 26, 'learning_rate': 0.1692252734973306, 'n_estimators': 750, 'colsample_bytree': 0.5011296167592567, 'subsample': 0.7606130136101464, 'reg_alpha': 0.1614918214989138, 'reg_lambda': 0.08739964218876393, 'min_child_samples': 159, 'min_split_gain': 0.1607167531255701}. Best is trial 0 with value: 1.3101856740643845.
[I 2026-08-27 16:40:06,042] Trial 2 finished with value: 1.5078704448512141 and parameters: {'max_depth': 9, 'num_leave

KeyboardInterrupt: 